# Sobrevivencia de negocios en la zona metropolitana de Guadalajara

¿Qué negocios de DENUE de noviembre de 2019 siguen en noviembre de 2024, y les va mejor a los que están junto a otros de su mismo giro?

Estudio aparte de Mapo (`docs/producto/sobrevivencia.md`). Lo que sale aquí describe; ninguna probabilidad por negocio o por zona pasa a Mapo, porque sería un puntaje (RNF5).

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from sobrevivencia import datos, graficas, modelos, rasgos

RAIZ = Path("../..")
EDICIONES = RAIZ / "mapo_core/data/denue_ediciones"
CLASES = RAIZ / "mapo_core/src/mapo_core/rubros/clases.csv"
TAXONOMIA = RAIZ / "mapo_core/src/mapo_core/rubros/taxonomia.csv"
VIVIENDA = RAIZ / "mapo_core/data/vivienda_gdl_v7.zip"
ZMG = ["039", "120", "098", "101", "097", "070", "051", "044", "124", "002"]
CAJA_ZMG = (20.2, 21.0, -103.9, -102.9)  # lat mín, lat máx, lon mín, lon máx
BASE, FIN, ACTUAL = "2019_11", "2024_11", "2026_05"
CONSECUTIVAS = ["2016_10", "2017_11", "2018_11", "2019_11", "2020_11", "2021_11",
                "2022_11", "2023_11", "2024_11", "2025_05", "2026_05"]

## $\color{#dda}{\text{1. Ediciones de DENUE}}$

Se bajan de la descarga masiva de INEGI (unos 35 MB cada una) a una carpeta gitignorada. Si ya están, no se vuelven a bajar.

In [2]:
rutas = {e: datos.bajar(e, EDICIONES) for e in CONSECUTIVAS}
len(rutas)

11

## $\color{#dda}{\text{2. Por qué de 2019 a 2024}}$

Entre ediciones normales sigue casi el 100 % de los negocios: INEGI actualiza sobre todo a los grandes. Las bajas de los chicos aparecen de golpe cuando entra un Censo Económico. La ventana con sentido va de un corte censal al siguiente: 2019_11 (Censos 2019) a 2024_11 (Censos 2024).

In [3]:
consecutivas = datos.persistencia(rutas)
consecutivas

,de,a,negocios,siguen
0,2016_10,2017_11,365550,99.7
1,2017_11,2018_11,367384,99.7
2,2018_11,2019_11,369177,61.6
3,2019_11,2020_11,373491,98.6
4,2020_11,2021_11,378665,99.3
5,2021_11,2022_11,378100,99.9
6,2022_11,2023_11,378439,99.0
7,2023_11,2024_11,378713,71.3
8,2024_11,2025_05,399075,97.9
9,2025_05,2026_05,399103,99.0


In [4]:
fig = graficas.px.line(consecutivas, x="a", y="siguen", markers=True,
                       title="Negocios de Jalisco que siguen en la edición siguiente")
fig.update_layout(xaxis_title="Edición de DENUE", yaxis_title="Siguen de la edición anterior (%)", yaxis_range=[50, 102])
fig.show()

## $\color{#dda}{\text{3. Tabla base}}$

Una fila por negocio con coordenadas de la edición 2019_11 en los diez municipios de la zona metropolitana. La etiqueta `sigue` es que su `id` esté en 2024_11. `clee` no sirve de llave: no existe antes de 2022 y cambia aunque el negocio siga.

`sigue_o_reregistrado` además cuenta como vivo al que desapareció pero tiene en 2024 un negocio nuevo a menos de 30 m con la misma rama SCIAN y el mismo nombre (el mismo negocio con otro `id`).

In [5]:
base, fin, actual = (datos.leer(rutas[e]) for e in (BASE, FIN, ACTUAL))
catalogo = datos.rubros(CLASES, actual)
df = datos.tabla_base(base, fin, ZMG, catalogo, CAJA_ZMG)
df["meses_alta"] = datos.meses_desde_alta(df["fecha_alta"], BASE)
en_zmg = base["cve_mun"].isin(ZMG)
print(f"Sin coordenadas o fuera de la zona: {en_zmg.sum() - len(df):,} de {en_zmg.sum():,}")
df[["sigue", "sigue_o_reregistrado"]].mean().mul(100).round(1).rename("%").to_frame().assign(negocios=len(df))

Sin coordenadas o fuera de la zona: 12 de 221,525


,%,negocios
sigue,67.7,221513
sigue_o_reregistrado,68.5,221513


In [6]:
df = df.join(rasgos.vecindad(df)).join(rasgos.contexto(df, VIVIENDA))
df["bloque"] = rasgos.bloques(df)
df["log_vecinos_mismo"] = np.log1p(df["vecinos_mismo_giro"])
df["log_vecinos_todos"] = np.log1p(df["vecinos_todos"])
df[["vecinos_mismo_giro", "vecinos_todos", "dist_mismo_giro_m", "meses_alta", *rasgos.CONTEXTO]].describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
vecinos_mismo_giro,221513.0,14.5,67.6,0.0,0.0,2.0,7.0,902.0
vecinos_todos,221513.0,162.5,339.5,0.0,51.0,87.0,142.0,2945.0
dist_mismo_giro_m,221513.0,205.1,551.9,0.0,26.9,84.1,207.2,27678.6
meses_alta,221513.0,55.0,49.2,0.0,0.0,59.0,112.0,112.0
viv_ha,219525.0,26.0,15.1,0.0,13.4,25.1,36.8,115.5
graproes,219740.0,10.8,2.0,0.0,9.2,10.4,12.1,15.6
pct_viv_auto,218643.0,55.4,16.9,0.0,44.3,51.7,64.7,100.0
dis_km,220158.0,10.1,5.9,4.1,6.3,8.5,11.6,40.6


## $\color{#dda}{\text{4. Tasas a simple vista}}$

In [7]:
t = modelos.tasas(df.assign(estrato=pd.Categorical(df["estrato"], datos.ESTRATOS, ordered=True)), "sigue", "estrato")
graficas.barras_tasa(t, "estrato", "Siguen en 2024, por personal ocupado en 2019", "Personal ocupado").show()

In [8]:
cortes = [-1, 0, 2, 5, 10, 25, np.inf]
nombres = ["0", "1 a 2", "3 a 5", "6 a 10", "11 a 25", "más de 25"]
df["vecinos_mismo_rango"] = pd.cut(df["vecinos_mismo_giro"], cortes, labels=nombres)
t = modelos.tasas(df, "sigue", "vecinos_mismo_rango")
graficas.barras_tasa(t, "vecinos_mismo_rango", "Siguen en 2024, por vecinos del mismo giro a 200 m",
                     "Negocios del mismo giro a menos de 200 m (2019)").show()

Las tasas crudas mezclan giros: los giros que se agrupan también pueden ser los que duran más. La sección siguiente compara cada negocio solo contra los de su giro.

## $\color{#dda}{\text{5. ¿Les va mejor a los que se juntan?}}$

Probabilidad lineal con efecto fijo por giro y errores agrupados por bloques de 2 km. Cada coeficiente es cuántos puntos porcentuales cambia la probabilidad de seguir, con todo lo demás igual. Los vecinos van en log: multiplicar los vecinos por e (2.7) suma el coeficiente.

In [9]:
NUMERICAS = ["log_vecinos_mismo", "log_vecinos_todos", "en_plaza", "semifijo", "meses_alta", *rasgos.CONTEXTO]
efecto = modelos.efecto_dentro_del_giro(df, "sigue", NUMERICAS)
efecto

,puntos,desde,hasta,p
log_vecinos_mismo,1.649,1.144,2.155,0.000
log_vecinos_todos,1.726,1.105,2.348,0.000
en_plaza,-2.610,-4.620,-0.601,0.011
semifijo,-2.867,-5.055,-0.680,0.010
meses_alta,0.180,0.174,0.185,0.000
viv_ha,-0.013,-0.050,0.025,0.511
viv_ha_falta,-9.396,-19.571,0.779,0.070
graproes,-1.575,-2.220,-0.929,0.000
graproes_falta,9.874,-3.645,23.394,0.152
pct_viv_auto,0.139,0.072,0.206,0.000


In [10]:
ETIQUETAS = {
    "log_vecinos_mismo": "Vecinos del mismo giro a 200 m (log)",
    "log_vecinos_todos": "Vecinos de cualquier giro a 200 m (log)",
    "en_plaza": "En plaza, mercado o edificio comercial",
    "semifijo": "Semifijo",
    "meses_alta": "Meses en DENUE antes de 2019",
    "graproes": "Escolaridad de la AGEB (años)",
    "pct_viv_auto": "Viviendas con auto en la AGEB (%)",
}
graficas.coeficientes(efecto, ETIQUETAS, "Cambio en la probabilidad de seguir, dentro del mismo giro").show()

**Con re-registros.** Si el efecto se debiera a negocios que INEGI volvió a registrar con otro `id`, cambiaría al contarlos como vivos.

In [11]:
modelos.efecto_dentro_del_giro(df, "sigue_o_reregistrado", NUMERICAS).loc[["log_vecinos_mismo", "log_vecinos_todos"]]

,puntos,desde,hasta,p
log_vecinos_mismo,1.819,1.269,2.368,0.0
log_vecinos_todos,1.875,1.252,2.498,0.0


**¿Distinto en los giros que se agrupan?** Con la taxonomía de Mapo (`rubros/taxonomia.csv`, calculada con DENUE 2026): un giro se agrupa si su distancia media al vecino del rubro es menos del 80 % de la de los sorteos. El efecto principal de ese rasgo lo absorbe el efecto fijo; la interacción dice si juntarse rinde distinto ahí.

In [12]:
tax = pd.read_csv(TAXONOMIA, dtype={"cve_ent": str})
se_agrupa = set(tax.loc[tax["agrupacion"] < np.log(0.8), "rubro"])
df["giro_se_agrupa"] = df["rubro"].isin(se_agrupa).astype(int)
df["vecinos_mismo_x_se_agrupa"] = df["log_vecinos_mismo"] * df["giro_se_agrupa"]
modelos.efecto_dentro_del_giro(df, "sigue", NUMERICAS + ["vecinos_mismo_x_se_agrupa"]).loc[
    ["log_vecinos_mismo", "vecinos_mismo_x_se_agrupa"]]

,puntos,desde,hasta,p
log_vecinos_mismo,0.987,0.380,1.594,0.001
vecinos_mismo_x_se_agrupa,1.084,0.323,1.845,0.005


## $\color{#dda}{\text{6. ¿Qué tanto se puede predecir?}}$

Validación por bloques espaciales (5 pliegues). La referencia es predecir la tasa del giro y nada más.

In [13]:
metricas, fuera = modelos.validar(df, "sigue", NUMERICAS)
metricas.groupby("modelo")[["auc", "brier"]].agg(["mean", "std"]).round(3)

auc         brier       
                            mean    std   mean    std
modelo                                               
logística                  0.668  0.001  0.202  0.002
solo el giro (en muestra)  0.605    NaN  0.211    NaN
árboles                    0.670  0.007  0.202  0.002

In [14]:
graficas.calibracion(df["sigue"], fuera["árboles"], "Calibración de los árboles, fuera de muestra").show()

## $\color{#dda}{\text{7. Lo que no dice}}$

- **Desaparecer de DENUE no es lo mismo que cerrar.** Un negocio que cambió de nombre y de giro, o que INEGI no encontró, cuenta como baja. Los re-registros con el mismo nombre se midieron (sección 5) y cambian poco.
- **Es asociación, no causa.** Un giro que se junta en un mercado puede durar más por el mercado, no por los vecinos; el efecto fijo por giro quita las diferencias entre giros, no las de lugar.
- **La pandemia está adentro.** 2019 a 2024 incluye 2020 y 2021: es la sobrevivencia de ese periodo, no de cualquiera.
- **Solo la zona metropolitana de Guadalajara**, con los negocios que tenían coordenadas.
- **Ninguna probabilidad pasa a Mapo.** Sería un puntaje por negocio o por zona (RNF5).